In [15]:
import numpy as np
import awkward as ak
import uproot
import vector
vector.register_awkward()


def read_file(
        filepath,
        max_num_particles=128,
        particle_features=['part_pt', 'part_eta', 'part_phi', 'part_energy'],
        jet_features=['jet_pt', 'jet_eta', 'jet_phi', 'jet_energy'],
        labels=['label_QCD', 'label_Hbb', 'label_Hcc', 'label_Hgg', 'label_H4q',
                'label_Hqql', 'label_Zqq', 'label_Wqq', 'label_Tbqq', 'label_Tbl']):
    """Loads a single file from the JetClass dataset.

    **Arguments**

    - **filepath** : _str_
        - Path to the ROOT data file.
    - **max_num_particles** : _int_
        - The maximum number of particles to load for each jet. 
        Jets with fewer particles will be zero-padded, 
        and jets with more particles will be truncated.
    - **particle_features** : _List[str]_
        - A list of particle-level features to be loaded. 
        The available particle-level features are:
            - part_px
            - part_py
            - part_pz
            - part_energy
            - part_pt
            - part_eta
            - part_phi
            - part_deta: np.where(jet_eta>0, part_eta-jet_p4, -(part_eta-jet_p4))
            - part_dphi: delta_phi(part_phi, jet_phi)
            - part_d0val
            - part_d0err
            - part_dzval
            - part_dzerr
            - part_charge
            - part_isChargedHadron
            - part_isNeutralHadron
            - part_isPhoton
            - part_isElectron
            - part_isMuon
    - **jet_features** : _List[str]_
        - A list of jet-level features to be loaded. 
        The available jet-level features are:
            - jet_pt
            - jet_eta
            - jet_phi
            - jet_energy
            - jet_nparticles
            - jet_sdmass
            - jet_tau1
            - jet_tau2
            - jet_tau3
            - jet_tau4
    - **labels** : _List[str]_
        - A list of truth labels to be loaded. 
        The available label names are:
            - label_QCD
            - label_Hbb
            - label_Hcc
            - label_Hgg
            - label_H4q
            - label_Hqql
            - label_Zqq
            - label_Wqq
            - label_Tbqq
            - label_Tbl

    **Returns**

    - x_particles(_3-d numpy.ndarray_), x_jets(_2-d numpy.ndarray_), y(_2-d numpy.ndarray_)
        - `x_particles`: a zero-padded numpy array of particle-level features 
                         in the shape `(num_jets, num_particle_features, max_num_particles)`.
        - `x_jets`: a numpy array of jet-level features
                    in the shape `(num_jets, num_jet_features)`.
        - `y`: a one-hot encoded numpy array of the truth lables
               in the shape `(num_jets, num_classes)`.
    """

    def _pad(a, maxlen, value=0, dtype='float32'):
        if isinstance(a, np.ndarray) and a.ndim >= 2 and a.shape[1] == maxlen:
            return a
        elif isinstance(a, ak.Array):
            if a.ndim == 1:
                a = ak.unflatten(a, 1)
            a = ak.fill_none(ak.pad_none(a, maxlen, clip=True), value)
            return ak.values_astype(a, dtype)
        else:
            x = (np.ones((len(a), maxlen)) * value).astype(dtype)
            for idx, s in enumerate(a):
                if not len(s):
                    continue
                trunc = s[:maxlen].astype(dtype)
                x[idx, :len(trunc)] = trunc
            return x

    table = uproot.open(filepath)['tree'].arrays()

    p4 = vector.zip({'px': table['part_px'],
                     'py': table['part_py'],
                     'pz': table['part_pz'],
                     'energy': table['part_energy']})
    table['part_pt'] = p4.pt
    table['part_eta'] = p4.eta
    table['part_phi'] = p4.phi

    x_particles = np.stack([ak.to_numpy(_pad(table[n], maxlen=max_num_particles)) for n in particle_features], axis=1)
    x_jets = np.stack([ak.to_numpy(table[n]).astype('float32') for n in jet_features], axis=1)
    y = np.stack([ak.to_numpy(table[n]).astype('int') for n in labels], axis=1)

    return x_particles, x_jets, y

In [16]:
x_particles, x_jet, y = read_file("/eos/user/m/mgarciam/val_5M/HToBB_120.root", particle_features=['part_px', 'part_py', 'part_pz', 'part_energy'],)

In [17]:
idx = 0
particles_idx_0 = x_particles[idx,:, :]
mask_particles = np.sum(particles_idx_0, axis=0)>0
graph = particles_idx_0[:,mask_particles]

In [37]:
import sys
import os

sys.path.append(os.path.abspath('/afs/cern.ch/work/m/mgarciam/private/DiffusionGeometry/'))  # Add parent directory to path
from diffusion_geometry import DiffusionGeometry
from diffusion_geometry.visualisation import *

import numpy as np

from plotly.subplots import make_subplots
from figures.generate_data import gen_3d_data, load_image_point_cloud


dg = DiffusionGeometry.from_point_cloud(graph[0:3,:].T, n_function_basis = 10)

# Pick an eigenfunction as an example
f = dg.function_space.zeros()
f.coeffs[6] = 1

camera = dict(eye=dict(x=-0.4, y=1.85, z=0.5))
plot_scatter_3d(graph[0:3,:].T, color = f.to_ambient(), camera=camera).show()


In [42]:
#calculate the gradient with respect to E as a function 
f_point_wise = graph[3,:].T
gradient_f = f.grad()

plot_quiver_3d(
    graph[0:3,:].T, 
    quiver=gradient_f.to_ambient(), 
    scale=10, 
    line_width=2,
    camera=camera
).show()

In [ ]:
# new multivector for gatr with energy gradient 
f_point_wise = fourmomenta[:,0]
gradient_f = f.grad()
gradient_f_ambient = gradient_f.to_ambient()
grad_E = embed_translation(gradient_f_ambient)
E_scalar = embed_scalar(fourmomenta[:,0])
p = embed_point(fourmomenta[:,1:])
multivectors =  p + E_scalar + grad_E

# new scalars 
dg = DiffusionGeometry.from_point_cloud(fourmomenta[:,1:], n_function_basis = 10)
outputs = []
for k in range(10):
    f = dg.function_space.zeros()
    f.coeffs[10] = 1
    scalars = f.to_ambient()
    outputs.append(scalars.view(-1,1))
scalars_eign = torch.stack(outputs, dim=1)




In [45]:
dg = DiffusionGeometry.from_point_cloud(graph[0:3,:].T, n_function_basis = 10)
f = dg.function_space.zeros()
f.coeffs[9] = 1
f.to_ambient()

array([-2.22746873e+00,  3.69967407e+00,  3.60508190e+00, -4.57415670e+00,
       -2.52932372e-02, -1.14578184e+00,  1.69020124e-01,  1.82617080e-01,
       -6.06970719e-02, -5.76121670e-02, -6.03131106e-02, -2.60386996e-03,
        1.15982875e-01,  9.15473676e-02, -8.40103272e-02, -7.86689116e-02,
        9.87298891e-02,  7.07351456e-02, -1.46946328e-01, -1.48470025e-01,
       -1.35816760e-01,  4.11244725e-02, -5.06556268e-02, -3.62403108e-02,
        9.51036330e-02,  1.16283237e-01, -3.29630230e-02, -5.85808580e-02,
       -5.77570146e-02, -5.13364504e-02, -2.22167195e-02,  3.01623865e-02,
        1.27710597e-01,  1.45834508e-01,  3.51774767e-02,  5.74889409e-02,
        9.28672795e-02,  1.13118441e-01,  4.36215568e-02,  5.45935332e-02,
        1.77489516e-02,  2.28340314e-02,  1.88294113e-02,  4.67341429e-02,
        7.78746728e-03, -3.01962090e-02, -4.45038125e-02, -5.78854819e-02,
       -6.67256447e-02, -6.22707086e-02, -8.59048669e-02, -8.53951949e-02,
       -7.76246507e-02])